# Finding Points of Interest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/03-points-of-interest.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Query points of interest (POIs) by category
- Understand all available POI categories
- Import custom POI data from CSV
- Combine POI queries with isochrones
- Analyze POI accessibility patterns

## Prerequisites

- Completed [01-Getting Started](01-getting-started.ipynb)
- Completed [02-Isochrone Analysis](02-isochrone-analysis.ipynb)

## Setup

In [ ]:
# Install SocialMapper from GitHub (latest version)
!pip install -q "socialmapper[routing] @ git+https://github.com/mihiarc/socialmapper.git"

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

In [ ]:
import os
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

import socialmapper
print(f"SocialMapper v{socialmapper.__version__}")

from socialmapper import get_poi, create_isochrone
print("Ready!")

## What is a Point of Interest?

A POI is any location that might be useful or interesting:
- Restaurants, cafes, bars
- Hospitals, pharmacies, clinics
- Schools, libraries, universities
- Grocery stores, supermarkets
- Parks, gyms, community centers

SocialMapper queries **OpenStreetMap** for POI data, which means coverage varies by region.

> **Note:** OSM data is community-contributed. Urban areas typically have better coverage than rural areas.

## Basic POI Query

In [ ]:
# Find POIs near a location
pois = get_poi(
    location="Portland, OR",
    limit=20
)

print(f"Found {len(pois)} points of interest:")
for poi in pois[:5]:
    print(f"  {poi['name']}: {poi['category']} ({poi['distance_km']:.2f} km)")

## POI Result Structure

Each POI contains detailed information from OpenStreetMap.

In [ ]:
# Get a single POI and examine its structure
pois = get_poi("Portland, OR", categories=["food_and_drink"], limit=1)
poi = pois[0]

print("POI structure:")
print(f"  name: {poi['name']}")
print(f"  category: {poi['category']}")
print(f"  lat: {poi['lat']}")
print(f"  lon: {poi['lon']}")
print(f"  distance_km: {poi['distance_km']:.3f}")
print(f"  address: {poi.get('address', 'N/A')}")
print(f"  tags: {poi['tags']}")

## POI Categories Reference

SocialMapper organizes POIs into **11 main categories**. Each category includes multiple OpenStreetMap tag values.

| Category | Description | Example POI Types |
|----------|-------------|------------------|
| `food_and_drink` | Eating and drinking | restaurant, cafe, bar, fast_food, bakery |
| `shopping` | Retail establishments | supermarket, convenience, mall, clothes |
| `education` | Educational facilities | school, university, library, kindergarten |
| `healthcare` | Medical services | hospital, clinic, pharmacy, dentist |
| `transportation` | Transit and parking | bus_station, train_station, parking, fuel |
| `recreation` | Leisure and sports | park, gym, cinema, museum, playground |
| `services` | Professional services | bank, post_office, police, lawyer |
| `accommodation` | Lodging | hotel, motel, hostel, camp_site |
| `religious` | Places of worship | church, mosque, temple, synagogue |
| `utilities` | Public utilities | toilets, drinking_water, recycling |

In [ ]:
# View all categories and their contents
from socialmapper.poi_categorization import POI_CATEGORY_MAPPING, get_poi_category_info

# Get category summary
info = get_poi_category_info()

print("POI Category Summary:")
print("=" * 50)
for category, values in POI_CATEGORY_MAPPING.items():
    print(f"\n{category}: {len(values)} types")
    print(f"  Examples: {', '.join(values[:5])}...")

In [ ]:
# Check if a category is valid
from socialmapper.poi_categorization import is_valid_category, get_category_values

# Valid category check
print(f"'healthcare' is valid: {is_valid_category('healthcare')}")
print(f"'invalid_cat' is valid: {is_valid_category('invalid_cat')}")

# Get all values for a category
healthcare_values = get_category_values('healthcare')
print(f"\nHealthcare includes {len(healthcare_values)} types:")
print(f"  {', '.join(healthcare_values[:10])}...")

## Filtering by Category

In [ ]:
# Find specific categories
location = "Portland, OR"

# Single category
healthcare = get_poi(location, categories=["healthcare"], limit=10)
print(f"Healthcare POIs: {len(healthcare)}")

# Multiple categories
food_and_shopping = get_poi(
    location,
    categories=["food_and_drink", "shopping"],
    limit=30
)
print(f"Food & shopping POIs: {len(food_and_shopping)}")

# Show breakdown
from collections import Counter
categories = Counter(p['category'] for p in food_and_shopping)
print("\nBreakdown:")
for cat, count in categories.most_common():
    print(f"  {cat}: {count}")

## The `validate_coords` Parameter

By default, `get_poi()` validates that returned POIs have valid coordinates. You can disable this for performance.

> **Tip:** Keep `validate_coords=True` (default) unless you're confident about your data quality.

In [ ]:
# Default: coordinates are validated
pois_validated = get_poi(
    location="Portland, OR",
    categories=["healthcare"],
    limit=20,
    validate_coords=True  # default
)
print(f"With validation: {len(pois_validated)} POIs")

# Without validation (slightly faster for large queries)
pois_unvalidated = get_poi(
    location="Portland, OR",
    categories=["healthcare"],
    limit=20,
    validate_coords=False
)
print(f"Without validation: {len(pois_unvalidated)} POIs")

## Travel-Time Bounded Search

Find POIs within a travel-time boundary. This internally creates an isochrone.

In [ ]:
# Find restaurants within 15-minute walk
walkable_food = get_poi(
    location="Portland, OR",
    categories=["food_and_drink"],
    travel_time=15,  # 15-minute walk
    limit=50
)

print(f"Food & drink within 15-min walk: {len(walkable_food)}")

# Show distance distribution
distances = [p['distance_km'] for p in walkable_food]
if distances:
    print(f"\nDistance range: {min(distances):.2f} - {max(distances):.2f} km")
    print(f"Average distance: {sum(distances)/len(distances):.2f} km")

## Importing Custom POI Data

Use `import_poi_csv()` to import your own POI data from a CSV file.

Required columns (configurable):
- Name field (default: `name`)
- Latitude field (default: `latitude`)
- Longitude field (default: `longitude`)
- Type field (default: `type`)

In [ ]:
from socialmapper import import_poi_csv

# Create a sample CSV
import csv

sample_pois = [
    {"name": "Main Library", "latitude": 45.5189, "longitude": -122.6793, "type": "library"},
    {"name": "Community Center", "latitude": 45.5221, "longitude": -122.6872, "type": "community"},
    {"name": "City Park", "latitude": 45.5152, "longitude": -122.7153, "type": "park"},
]

with open("my_pois.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "latitude", "longitude", "type"])
    writer.writeheader()
    writer.writerows(sample_pois)

print("Created my_pois.csv")

In [ ]:
# Import the custom POIs
custom_pois = import_poi_csv(
    csv_path="my_pois.csv",
    name_field="name",
    lat_field="latitude",
    lon_field="longitude",
    type_field="type"
)

print(f"Imported {len(custom_pois)} custom POIs:")
for poi in custom_pois:
    print(f"  - {poi['name']} ({poi['category']}): ({poi['lat']}, {poi['lon']})")

## Healthcare Access Analysis

In [ ]:
location = "Portland, OR"

print(f"Healthcare Access in {location}:")
print("=" * 40)

# Get all healthcare POIs
healthcare = get_poi(location, categories=["healthcare"], limit=30)

print(f"\nTotal healthcare facilities: {len(healthcare)}")

# Show nearest facilities
print("\nNearest healthcare facilities:")
for h in sorted(healthcare, key=lambda x: x['distance_km'])[:5]:
    print(f"  {h['name']}: {h['distance_km']:.2f} km")

## Food Desert Identification

Check if an area has adequate grocery access (a key indicator for food deserts).

In [ ]:
def check_food_access(location, travel_time=15):
    """Check if an area has adequate grocery access."""
    
    groceries = get_poi(
        location,
        categories=["shopping"],
        travel_time=travel_time,
        limit=50
    )
    
    print(f"Food Access Analysis: {location}")
    print("=" * 40)
    print(f"Grocery stores within {travel_time}-min walk: {len(groceries)}")
    
    if groceries:
        distances = [g['distance_km'] for g in groceries]
        print(f"Nearest store: {min(distances):.2f} km")
    
    # Assessment based on USDA criteria
    if len(groceries) < 2:
        status = "POTENTIAL FOOD DESERT"
    elif len(groceries) < 5:
        status = "LIMITED ACCESS"
    else:
        status = "GOOD ACCESS"
    
    print(f"\nAssessment: {status}")
    return groceries

# Test different areas
check_food_access("Portland, OR")

## Combining POIs with Isochrones

In [ ]:
from shapely.geometry import shape, Point

# Create isochrone
location = "Portland, OR"
isochrone = create_isochrone(location, travel_time=10, travel_mode="walk")
polygon = shape(isochrone['geometry'])

# Get POIs in a larger area
all_pois = get_poi(location, categories=["food_and_drink"], limit=100)

# Filter to only those inside the isochrone
accessible = []
for poi in all_pois:
    point = Point(poi['lon'], poi['lat'])
    if polygon.contains(point):
        accessible.append(poi)

print(f"Food & drink within 10-min walk: {len(accessible)}")
print(f"Total found: {len(all_pois)}")
print(f"Accessibility rate: {len(accessible)/len(all_pois)*100:.1f}%")

## Data Quality Notes

> **Warning:** OSM data quality varies significantly by location.

### Coverage Considerations

| Area Type | Typical Coverage | Notes |
|-----------|------------------|-------|
| Major US cities | Good | Most POIs mapped |
| Suburban areas | Moderate | Some gaps possible |
| Rural areas | Variable | May have significant gaps |
| International | Varies | Europe excellent, others vary |

### Best Practices

1. **Verify critical results** - For high-stakes analysis, cross-check with other sources
2. **Account for unmapped POIs** - Real count may be higher than OSM shows
3. **Consider recency** - Some POIs may be outdated
4. **Use multiple categories** - A "grocery store" might be tagged as `supermarket` or `convenience`

In [ ]:
# Example: Check data quality by examining tags
pois = get_poi("Portland, OR", categories=["food_and_drink"], limit=20)

print("POI Data Quality Check:")
print("=" * 50)

has_website = sum(1 for p in pois if p['tags'].get('website'))
has_phone = sum(1 for p in pois if p['tags'].get('phone'))
has_opening_hours = sum(1 for p in pois if p['tags'].get('opening_hours'))

print(f"POIs checked: {len(pois)}")
print(f"  With website: {has_website} ({has_website/len(pois)*100:.0f}%)")
print(f"  With phone: {has_phone} ({has_phone/len(pois)*100:.0f}%)")
print(f"  With hours: {has_opening_hours} ({has_opening_hours/len(pois)*100:.0f}%)")

## Amenity Summary Report

In [ ]:
def generate_amenity_report(location):
    """Generate a comprehensive amenity report for a location."""
    
    categories_to_check = {
        "Food": ["food_and_drink"],
        "Healthcare": ["healthcare"],
        "Education": ["education"],
        "Recreation": ["recreation"]
    }
    
    print(f"\nAmenity Report: {location}")
    print("=" * 50)
    
    for group, cats in categories_to_check.items():
        pois = get_poi(location, categories=cats, limit=50)
        
        if pois:
            nearest = min(p['distance_km'] for p in pois)
            print(f"\n{group}:")
            print(f"  Total found: {len(pois)}")
            print(f"  Nearest: {nearest:.2f} km")
            
            # Show top 3
            for p in sorted(pois, key=lambda x: x['distance_km'])[:3]:
                print(f"    - {p['name']}: {p['distance_km']:.2f} km")
        else:
            print(f"\n{group}: None found nearby")

# Generate report
generate_amenity_report("Portland, OR")

## Troubleshooting

### Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| `InvalidPOICategoryError` | Invalid category name | Check spelling, use valid category |
| Empty results | No OSM data for area | Try broader search, different location |
| Missing POIs | OSM coverage gaps | Cross-check with other sources |
| Slow queries | Large search area | Reduce `limit` or use `travel_time` |
| Duplicate POIs | Multiple OSM entries | Filter by unique coordinates |

In [ ]:
from socialmapper import InvalidPOICategoryError

# Handle invalid category errors
try:
    pois = get_poi(
        location="Portland, OR",
        categories=["invalid_category"]
    )
except InvalidPOICategoryError as e:
    print(f"Error: {e}")
    print(f"\nValid categories:")
    for cat in e.valid_categories:
        print(f"  - {cat}")

## Next Steps

Continue with:

- **[Census Data](04-census-data.ipynb)** - Add demographic context to your POI analysis
- **[Mapping & Visualization](05-mapping-visualization.ipynb)** - Visualize POI locations on maps
- **[Complete Workflow](06-complete-workflow.ipynb)** - Full analysis examples